# Torch vs ONNX performance

In [53]:
import torch
import onnx
import onnxruntime
import lenet
import numpy as np

torch.manual_seed(42)

device = torch.accelerator.current_accelerator() if torch.accelerator.is_available() else "cpu"
print(f"Using device: {device}")

Using device: cpu


## Test dataset

In [14]:
sample = torch.load(lenet.sample_filename, map_location=device)
X = sample["images"]
y = sample["labels"]

X.shape, y.shape

(torch.Size([64, 1, 28, 28]), torch.Size([64]))

In [37]:
X[0].reshape(1, 1, 28, 28)

tensor([[[[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000],
          [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
           0.0000, 0.0000, 0.0000, 0.0000, 

In [20]:
lenet_model = lenet.LeNetModel().to(device)
lenet_model.load_state_dict(torch.load(lenet.torch_filename, map_location=device))
lenet_model.eval()


LeNetModel(
  (lenet_stack): Sequential(
    (0): Conv2d(1, 12, kernel_size=(3, 3), stride=(1, 1), padding=(2, 2))
    (1): ReLU()
    (2): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (3): Conv2d(12, 32, kernel_size=(3, 3), stride=(1, 1))
    (4): ReLU()
    (5): AvgPool2d(kernel_size=2, stride=2, padding=0)
    (6): Flatten(start_dim=1, end_dim=-1)
    (7): Linear(in_features=1152, out_features=120, bias=True)
    (8): ReLU()
    (9): Linear(in_features=120, out_features=84, bias=True)
    (10): Sigmoid()
    (11): Linear(in_features=84, out_features=10, bias=True)
  )
)

In [64]:
onnxruntime_session = onnxruntime.InferenceSession(lenet.onnx_filename)

logits = torch.zeros((len(X), 10))

with torch.no_grad():
    for idx in range(len(y)):
        img = X[idx]
        label = y[idx]
        pred_onnx = onnxruntime_session.run(None, {"images": img.unsqueeze(0).numpy()})[0].reshape(-1)
        pred_torch = lenet_model(img.reshape(1, 1, 28, 28)).numpy().reshape(-1)
        logits[idx] = torch.from_numpy(pred_torch)

        if idx == 0:
            print(f"label: {label}")
            print(f"torch: {pred_torch}")
            print(f"onnx: {pred_onnx}")

        print(np.abs(pred_onnx - pred_torch).max())
        if pred_torch.argmax() != pred_onnx.argmax():
            print(f"Mismatch prediction at sample {idx}: label={label}, pred_torch={pred_torch.argmax()}, pred_onnx={pred_onnx.argmax()}")

torch.save({"logits": logits}, "lenet_torch_logits.pt")


label: 5
torch: [-3.0544376  -1.3292665  -1.5134838  -0.63088924 -7.112322   12.542642
 -1.1275438   6.0617504  -1.3873163   1.3627545 ]
onnx: [-3.0544376 -1.3292674 -1.513484  -0.630889  -7.1123204 12.542641
 -1.1275439  6.0617495 -1.3873167  1.3627546]
1.4305115e-06
2.861023e-06
2.861023e-06
1.9073486e-06
1.1920929e-06
1.3113022e-06
9.536743e-07
1.9073486e-06
1.9073486e-06
1.9073486e-06
1.9073486e-06
2.861023e-06
1.4305115e-06
9.536743e-07
9.536743e-07
1.9073486e-06
9.536743e-07
1.9073486e-06
2.3841858e-06
1.66893e-06
9.536743e-07
1.4305115e-06
3.8146973e-06
1.0728836e-06
9.536743e-07
1.4305115e-06
1.1920929e-06
1.9073486e-06
1.9073486e-06
1.1920929e-06
1.4305115e-06
1.9073486e-06
1.9073486e-06
1.9073486e-06
1.7285347e-06
1.9073486e-06
2.6226044e-06
9.536743e-07
3.8146973e-06
1.1920929e-06
1.1920929e-06
1.4305115e-06
9.536743e-07
1.9073486e-06
1.9073486e-06
1.4305115e-06
1.4305115e-06
1.9073486e-06
1.9073486e-06
1.9073486e-06
2.861023e-06
9.536743e-07
9.536743e-07
1.66893e-06
1.90734